## Exercise 2 Text Similarity from Embeddings

In [1]:
#Not needed if your kernel already has gensim, this is for my local jupyter instance
#Cell can be deleted
!uv add gensim

Resolved 61 packages in 24ms
Checked 54 packages in 18ms


In [2]:
import gensim
from gensim import utils
import pandas as pd

In [3]:
# An iterator that yields sentences (lists of str).
class MyCorpus:
    def __iter__(self):
        #corpus=open("/Users/aileenjoanvicente/Course_Materials/CMSC176 Natural Language Processing/demo/embeddings/ceb-clean.txt")
        corpus=open("./ceb-clean.txt")
        
        for line in corpus:
            # assume there's one document per line, tokens separated by whitespace
            yield utils.simple_preprocess(line)

In [52]:
# Train Word2Vec Model
sentences = MyCorpus()
w2v_model = gensim.models.Word2Vec(sentences=sentences, vector_size=200)


In [53]:
# a. What is the count of unique tokens? Display answer.

# 1. Unique tokens in the trained Word2Vec model:
print(f"Unique tokens in Word2Vec model: {len(w2v_model.wv)}")

# 2. (Optional) Unique tokens in the raw text:
raw_vocab = set()
for sent in sentences:
    raw_vocab.update(sent)
print(f"Unique tokens in raw corpus: {len(raw_vocab)}")

Unique tokens in Word2Vec model: 9976
Unique tokens in raw corpus: 25929


# Word Similarity #
Target Words: 
1. gugma
2. kalinaw
3. malinawon
4. kagawasan
5. panimalay
6. kaluwasan
7. luwas
8. dios
9. babaye

In [54]:
# b. What are the top 10 similar words for each target word? Describe the kind of 
#    similarity produced by the model?
# w2v_model.wv.most_similar(target)

target_words = [
    "gugma",
    "kalinaw",
    "malinawon",
    "kagawasan",
    "panimalay",
    "kaluwasan",
    "luwas",
    "dios",
    "babaye"
]

print("Score close to 1.0: Identical direction (identical meaning)")
print("Score close to 0.0: Completely unrelated/orthogonal")
print("Score close to -1.0: Exact opposites\n")

for word in target_words:
    print(f"Top 10 words similar to '{word}':")
    if word in w2v_model.wv:
        similar_words = w2v_model.wv.most_similar(word, topn=10)
        itr = 1
        for similar_word, cosine_similarity_score in similar_words:
            print(f"{itr}. {similar_word}: {cosine_similarity_score:.4f}")
            itr+=1
        print()
    else:
        print(f"Word '{word}' not found in vocabulary.\n")

# What I realized here is that, words with cosine similarity scores closest to 1.0
# are the most similar to each other. This does not only mean
# they are synonyms. The model groups words that share the same
# root, belong to the same topic (like love, mercy, and faith
# in a biblical text), or usually appear in the same type of
# sentence. Because of this, even antonyms can score high
# since they tend to appear in similar contexts.
    

Score close to 1.0: Identical direction (identical meaning)
Score close to 0.0: Completely unrelated/orthogonal
Score close to -1.0: Exact opposites

Top 10 words similar to 'gugma':
1. paglaum: 0.7957
2. pagtoo: 0.7864
3. balus: 0.7814
4. kamatuoran: 0.7782
5. pagkadautan: 0.7240
6. paghigugma: 0.7204
7. hunahuna: 0.7140
8. kaayohan: 0.7095
9. pagkamatarung: 0.7009
10. tinguha: 0.6959

Top 10 words similar to 'kalinaw':
1. pahulay: 0.6571
2. panalangin: 0.6175
3. kaluwasan: 0.5926
4. gugma: 0.5844
5. paglaum: 0.5828
6. grasya: 0.5802
7. kalibutan: 0.5584
8. presencia: 0.5571
9. manlalaglag: 0.5544
10. dayong: 0.5541

Top 10 words similar to 'malinawon':
1. nakapito: 0.7488
2. nagpagula: 0.7483
3. labihan: 0.7459
4. mogitib: 0.7432
5. mapainubsanon: 0.7420
6. igapadayag: 0.7414
7. mingsaka: 0.7376
8. nagapadayag: 0.7364
9. mala: 0.7357
10. nangandoy: 0.7348

Top 10 words similar to 'kagawasan':
1. suhol: 0.7376
2. balus: 0.6992
3. kasibut: 0.6967
4. tinagoan: 0.6909
5. pahamangno: 0.67

# Word Analogy #

[model.most_similar(positive=[y1, x2], negative=[x1])]<br>
Evaluate the model using word analogies. Example analogies are provided for you.
You may look for other analogies to use as test cases.
1. synonym             -- amahan:lalake::inahan:?
2. antonymity          -- amahan:inahan::lalake:?
3. part-whole          -- bahay:atip::kotse:?                        
4. superclass-subclass -- hayop:aso::tao:?
5. used-for            -- gupit:gunting::sulat:? 

In [ ]:
# c. Describe the ability of the model to capture synonyms. Use 2-3 example analogies.

synonym_analogies = {
    "amahan : lalake :: inahan : ?": (['lalake', 'inahan'], ['amahan'], "babaye"),
    "langit : kalangitan :: yuta : ?": (['kalangitan', 'yuta'], ['langit'], "kalibutan"),
    "bana : lalake :: asawa : ?": (['lalake', 'asawa'], ['bana'], "babaye")
}

for analogy, (pos, neg, expected) in synonym_analogies.items():
    print(f"Analogy: {analogy} (Expected: '{expected}')")
    synonyms = w2v_model.wv.most_similar(positive=pos, negative=neg, topn=10)
    
    itr = 1
    for synonym, cosine_similarity_score in synonyms:
        print(f"{itr}. {synonym}: {cosine_similarity_score:.4f}")
        itr+=1
    print()

# I observed that the model does not perform well in
# analogy tasks. The expected answers either ranked low
# or did not appear in the top 10 at all. It does not
# truly understand word relationships, it only relies
# on contextual co-occurrence. This is evident where
# "lalaki" ranked first instead of "babaye" simply
# because "lalaki" and "lalake" are spelling variations
# of the same word, creating a bias toward it rather
# than reasoning through the actual analogy/semantic.


Analogy: amahan : lalake :: inahan : ? (Expected: 'babaye')
1. lalaki: 0.7363
2. babayeng: 0.6966
3. magpaliwat: 0.6420
4. babaye: 0.6363
5. sallum: 0.6322
6. uyoan: 0.6090
7. gianak: 0.5901
8. ater: 0.5890
9. asawa: 0.5885
10. boseth: 0.5865

Analogy: langit : kalangitan :: yuta : ? (Expected: 'kalibutan')
1. kapatagan: 0.6429
2. mananap: 0.5742
3. pagatuktokon: 0.5721
4. kayutaan: 0.5668
5. mapintas: 0.5614
6. kabukiran: 0.5550
7. mohubad: 0.5495
8. langgam: 0.5481
9. kakahoyan: 0.5301
10. sapa: 0.5169

Analogy: bana : lalake :: asawa : ? (Expected: 'babaye')
1. coath: 0.6674
2. ruben: 0.6596
3. babaye: 0.6467
4. zebedeo: 0.6450
5. levi: 0.6410
6. simeon: 0.6357
7. gad: 0.6320
8. manases: 0.6319
9. heman: 0.6237
10. asaph: 0.6113



In [56]:
# d. Describe the ability of the model to capture antonyms. Use 2-3 example analogies.

antonym_analogies = {
    "amahan : inahan :: lalake : ?": (['inahan', 'lalake'], ['amahan'], "babaye"),
    "adlaw : gabii :: kahayag : ?": (['gabii', 'kahayag'], ['adlaw'], "kangitngit"),
    "kasadpan : sidlakan :: habagatan : ?": (['sidlakan', 'habagatan'], ['kasadpan'], "amihanan")
}

for analogy, (pos, neg, expected) in antonym_analogies.items():
    print(f"Analogy: {analogy} (Expected: '{expected}')")
    antonyms = w2v_model.wv.most_similar(positive=pos, negative=neg, topn=10)
    
    itr = 1
    for antonym, cosine_similarity_score in antonyms:
        print(f"{itr}. {antonym}: {cosine_similarity_score:.4f}")
        itr += 1
    print()

# I observed that the model performs significantly better
# with antonyms than synonyms, especially for highly paired
# concepts like light/dark (kahayag -> kangitngit) and
# directional opposites (habagatan -> amihanan), where the
# expected answers ranked directly at #1. This is because
# antonymous pairs frequently appear in parallel and
# contrasting sentence structures throughout the biblical text.


Analogy: amahan : inahan :: lalake : ? (Expected: 'babaye')
1. lalaki: 0.7363
2. babayeng: 0.6966
3. magpaliwat: 0.6420
4. babaye: 0.6363
5. sallum: 0.6322
6. uyoan: 0.6090
7. gianak: 0.5901
8. ater: 0.5890
9. asawa: 0.5885
10. boseth: 0.5865

Analogy: adlaw : gabii :: kahayag : ? (Expected: 'kangitngit')
1. kangitngit: 0.7232
2. kainit: 0.6282
3. mopanaw: 0.6100
4. pagsubang: 0.6033
5. pagsilang: 0.5919
6. tun: 0.5859
7. kagabhion: 0.5836
8. kasamok: 0.5798
9. subangan: 0.5788
10. ulan: 0.5693

Analogy: kasadpan : sidlakan :: habagatan : ? (Expected: 'amihanan')
1. amihanan: 0.8928
2. silangan: 0.8614
3. ubay: 0.8332
4. tungasan: 0.8303
5. paingon: 0.8236
6. cades: 0.8049
7. atbang: 0.7964
8. nagaatubang: 0.7942
9. jerico: 0.7941
10. padulong: 0.7924



In [57]:
# e. Describe the ability of the model to capture part-whole relationship. Use 2-3 example analogies.

part_whole_analogies = {
    "kahoy : sanga :: lawas : ?": (['sanga', 'lawas'], ['kahoy'], "ulo"),
    "balay : pultahan :: lungsod : ?": (['pultahan', 'lungsod'], ['balay'], "ganghaan"),
    "lawas : ulo :: balay : ?": (['ulo', 'balay'], ['lawas'], "atop")
}

for analogy, (pos, neg, expected) in part_whole_analogies.items():
    print(f"Analogy: {analogy} (Expected: '{expected}')")
    parts = w2v_model.wv.most_similar(positive=pos, negative=neg, topn=10)
    
    itr = 1
    for part, cosine_similarity_score in parts:
        print(f"{itr}. {part}: {cosine_similarity_score:.4f}")
        itr += 1
    print()

# I observed that the model has moderate to weak ability in
# capturing part-whole relationships. For body parts, it
# retrieved related physical parts like 'panit', 'ulo', and 'buhok',
# but failed to place the exact expected word at #1. For physical
# structures like house/city, it predicted related location words
# rather than strictly identifying structural components.


Analogy: kahoy : sanga :: lawas : ? (Expected: 'ulo')
1. panit: 0.6448
2. sakit: 0.6338
3. panagway: 0.6317
4. ngipon: 0.6209
5. hawak: 0.6183
6. bukog: 0.6141
7. buhok: 0.6130
8. unod: 0.5994
9. lapa: 0.5899
10. sanla: 0.5884

Analogy: balay : pultahan :: lungsod : ? (Expected: 'ganghaan')
1. balangay: 0.7610
2. atbang: 0.7215
3. kiliran: 0.6935
4. duol: 0.6802
5. el: 0.6681
6. semes: 0.6673
7. daplin: 0.6663
8. sidlakan: 0.6661
9. pikas: 0.6647
10. tamboanan: 0.6642

Analogy: lawas : ulo :: balay : ? (Expected: 'atop')
1. sungkod: 0.5268
2. ganghaan: 0.5167
3. lawak: 0.5014
4. pultahan: 0.4716
5. balongbalong: 0.4617
6. bilanggoan: 0.4570
7. trono: 0.4558
8. sawang: 0.4521
9. taming: 0.4443
10. tukmaan: 0.4394



In [58]:
# f. Describe the ability of the model to capture superclass-subclass relationship. Use 2-3 example analogies.

superclass_subclass_analogies = {
    "mananap : karnero :: langgam : ?": (['karnero', 'langgam'], ['mananap'], "salampati"),
    "mananap : leon :: langgam : ?": (['leon', 'langgam'], ['mananap'], "agila"),
    "kahoy : igos :: tanum : ?": (['igos', 'tanum'], ['kahoy'], "trigo")
}

for analogy, (pos, neg, expected) in superclass_subclass_analogies.items():
    print(f"Analogy: {analogy} (Expected: '{expected}')")
    subclasses = w2v_model.wv.most_similar(positive=pos, negative=neg, topn=10)
    
    itr = 1
    for subclass, cosine_similarity_score in subclasses:
        print(f"{itr}. {subclass}: {cosine_similarity_score:.4f}")
        itr += 1
    print()

# I observed that the model shows good performance when
# identifying subclasses within biblical categories. For example,
# shifting from animals (mananap) to birds (langgam) successfully
# ranked specific biblical birds like 'salampati' (dove) and 'agila' (eagle)
# in the top 3. The model groups species well when both categories
# are prominent in the corpus.


Analogy: mananap : karnero :: langgam : ? (Expected: 'salampati')
1. toril: 0.7372
2. nating: 0.7366
3. baka: 0.7291
4. lakeng: 0.7156
5. kanding: 0.6897
6. carnero: 0.6715
7. sungay: 0.6669
8. salampati: 0.6647
9. lobo: 0.6614
10. nati: 0.6592

Analogy: mananap : leon :: langgam : ? (Expected: 'agila')
1. salampati: 0.7979
2. oso: 0.7639
3. alimpulos: 0.7554
4. agila: 0.7541
5. lagsaw: 0.7372
6. bitin: 0.7295
7. ligid: 0.7227
8. dulon: 0.7166
9. querubin: 0.6825
10. sulo: 0.6809

Analogy: kahoy : igos :: tanum : ? (Expected: 'trigo')
1. tinulo: 0.7035
2. agdaha: 0.7024
3. sudlanang: 0.7007
4. sabton: 0.6966
5. umaabut: 0.6950
6. rubi: 0.6941
7. nagakalainlaing: 0.6937
8. tipik: 0.6917
9. talabong: 0.6916
10. mamumuno: 0.6826



In [59]:
# g. Describe the ability of the model to capture used-for relationship. Use 2-3 example analogies.

used_for_analogies = {
    "gubat : espada :: panaw : ?": (['espada', 'panaw'], ['gubat'], "sugkod"),
    "sulat : basahon :: kalan : ?": (['basahon', 'kalan'], ['sulat'], "tinapay"),
    "gubat : bangkaw :: uma : ?": (['bangkaw', 'uma'], ['gubat'], "daro")
}

for analogy, (pos, neg, expected) in used_for_analogies.items():
    print(f"Analogy: {analogy} (Expected: '{expected}')")
    tools = w2v_model.wv.most_similar(positive=pos, negative=neg, topn=10)
    
    itr = 1
    for tool, cosine_similarity_score in tools:
        print(f"{itr}. {tool}: {cosine_similarity_score:.4f}")
        itr += 1
    print()

# I observed that the model performs poorly on 'used-for'
# relationships. The expected tools or instruments (such as 'sugkod'
# or 'daro') did not rank within the top 10. Word2Vec relies strictly
# on word co-occurrence within a small window, so it cannot reason
# about functional utility (what tool is used to perform an action).


Analogy: gubat : espada :: panaw : ? (Expected: 'sugkod')
1. tiyan: 0.6120
2. paggikan: 0.6007
3. liog: 0.5966
4. pagpanaw: 0.5933
5. paghimugso: 0.5857
6. ilong: 0.5808
7. yugo: 0.5756
8. gibantayan: 0.5712
9. pagaadtoan: 0.5667
10. lakang: 0.5615

Analogy: sulat : basahon :: kalan : ? (Expected: 'tinapay')
1. pagapakasad: 0.6168
2. pagapun: 0.6123
3. kinabubut: 0.6070
4. magamalig: 0.5923
5. makapalig: 0.5854
6. maminatud: 0.5820
7. gidak: 0.5789
8. pagakan: 0.5786
9. mamalig: 0.5762
10. pakasad: 0.5709

Analogy: gubat : bangkaw :: uma : ? (Expected: 'daro')
1. singsing: 0.7197
2. gitakdoan: 0.7141
3. sidsid: 0.7098
4. bakus: 0.7093
5. kahoyng: 0.6902
6. copa: 0.6869
7. kilid: 0.6843
8. tumoy: 0.6785
9. kolintas: 0.6782
10. bulak: 0.6777



In [60]:
#example analogy
w2v_model.wv.most_similar(positive=['amahan','lalaki'], negative=['inahan'])

[('dumuloong', 0.5425938963890076),
 ('lalake', 0.5246033072471619),
 ('kaigsoonan', 0.5220335125923157),
 ('babaye', 0.5122529864311218),
 ('pinalangga', 0.4939645826816559),
 ('panimalay', 0.470908522605896),
 ('asawa', 0.46279338002204895),
 ('lumalangyaw', 0.45797333121299744),
 ('igsoon', 0.45236924290657043),
 ('ilo', 0.4393821656703949)]